# ReviewPulse: turn hundreds of customer reviews into clear actions

ReviewPulse reads every online review of a business and tells the owner, in a few minutes:
what customers love, what frustrates them, which complaints were never answered, and how to reply.

**How to run (about 5 minutes):**
1. `Runtime > Change runtime type > T4 GPU` (free), then **Save**.
2. `Runtime > Run all`.
3. Scroll to the last cell and open the public `gradio.live` link to use the app.

Pipeline: multilingual sentiment model → zero-shot theme detection → statistics → small open-source LLM (summary and reply drafts). All models are free and open source, downloaded from Hugging Face. No API key is needed.

## 1. Check the GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU ready:", torch.cuda.get_device_name(0))
else:
    print("No GPU. The app works but text generation will be very slow.\n"
          "Fix: Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")

## 2. Get the code and install the dependencies (about 1 minute)

In [ ]:
import os, sys
REPO_URL = "https://github.com/Monka31/reviewpulse.git"

if os.path.exists("/content"):                    # Google Colab
    # Cloned under a different name than the Python package "reviewpulse", to avoid a name clash
    REPO_DIR = "/content/reviewpulse_app"
    if not os.path.exists(os.path.join(REPO_DIR, "app.py")):
        !git clone -q {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !git pull -q                                  # always use the latest version from GitHub
!pip install -q -r requirements.txt

sys.path.insert(0, os.getcwd())
for name in [m for m in list(sys.modules) if m == "reviewpulse" or m.startswith("reviewpulse.")]:
    del sys.modules[name]                         # forget any stale import from a previous run
import reviewpulse
assert hasattr(reviewpulse, "run_pipeline"), "Code not found: run this cell again"
print("Ready in", os.getcwd())

## 3. Choose the reviews to analyze

The `data/` folder contains the datasets used in the demo. To analyze your own file, set `UPLOAD = True`.
Any CSV or Excel file works as long as one column holds the review text (columns are detected automatically: see the user guide).

In [ ]:
import glob
print("Available datasets:\n " + "\n ".join(sorted(glob.glob("data/*.csv"))))

DATA_FILE = "data/reviews_ESSEC_Cergy.csv"   # or "data/demo_restaurant_synthetic.csv", "data/reviews_<restaurant>.csv"
BUSINESS_NAME = "ESSEC Business School (Cergy campus)"   # None = read from the file
BUSINESS_TYPE = "School / University"  # "Restaurant", "School / University", "Hotel", "Retail store", "Generic business"
LANGUAGE = "English"          # language of the AI summary ("English" or "French")
N_REPLIES = 10                # reply drafts for the most recent unanswered negative reviews
MAX_REVIEWS = 500

UPLOAD = False
if UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    DATA_FILE = next(iter(uploaded))

## 4. Run the analysis
The first run downloads the three models (about 2 minutes). Next runs reuse them.

In [ ]:
import time
import reviewpulse as rp

start, last = time.time(), [None]
def show_progress(fraction, desc=""):
    if desc != last[0]:
        print(f"[{time.time() - start:4.0f}s] {desc}")
        last[0] = desc

result = rp.run_pipeline(DATA_FILE, business_name=BUSINESS_NAME, business_type=BUSINESS_TYPE,
                         max_reviews=MAX_REVIEWS, language=LANGUAGE, n_replies=N_REPLIES,
                         progress=show_progress)
print(f"\nDone: {result.stats['n_reviews']} reviews in {time.time() - start:.0f}s")

## 5. Results

In [ ]:
from IPython.display import HTML, display
from reviewpulse import dashboard
# The same dashboard as in the app (fonts load from Google Fonts)
display(HTML(dashboard.FONTS_LINK + "<style>" + dashboard.CSS + "</style>" + dashboard.render_dashboard(result)))

The cells below show the same results with interactive Plotly charts and tables.

In [ ]:
from IPython.display import Markdown, display
from reviewpulse import charts

s = result.stats
display(Markdown(f"""### {result.business_name}
**{s['n_reviews']} reviews** · average rating **{s.get('avg_rating') or s['avg_predicted_stars']}/5** ·
**{s['pct_positive']:.0f}%** positive · **{s['pct_negative']:.0f}%** negative""" +
    (f""" · **{s['n_unanswered_negative']}** negative reviews without an answer""" if 'reply_rate' in s else "")))
display(Markdown(result.summary))
charts.theme_chart(result).show()
result.theme_table

In [ ]:
charts.rating_chart(result).show()
trend = charts.trend_chart(result)
if trend: trend.show()

### Unanswered negative reviews and AI reply drafts

In [ ]:
for idx, reply in result.replies.items():
    row = result.reviews.loc[idx]
    display(Markdown(f"**Review #{idx}** ({row['rating']:.0f}★)\n\n> {row['text']}\n\n**Suggested reply:**\n\n{reply}\n\n---"))
if not result.replies:
    print("No unanswered negative review (or no owner-reply column in the file).")

### Model evaluation
Customers' star ratings are free ground-truth labels: we measure how often the AI agrees with them.

In [ ]:
from reviewpulse.evaluation import evaluate, evaluation_markdown
ev = evaluate(result.reviews)
display(Markdown(evaluation_markdown(ev)))
if ev:
    charts.confusion_chart(ev["confusion_classes"], "Confusion matrix (sentiment classes)").show()

## 6. Export a permanent HTML report (GitHub Pages)
This writes `docs/reports/<business>.html` and updates `docs/index.html`. Download the zip, then upload the `docs` folder content to the GitHub repository: the report gets a permanent public link (see the user guide).

In [ ]:
path = rp.export_report(result, out_dir="docs", max_replies=N_REPLIES)
print("Report written to", path)
!cd docs && zip -qr ../site.zip . && cd ..
try:
    from google.colab import files
    files.download("site.zip")
except ImportError:
    print("site.zip created (download it from the file panel)")

## 7. Launch the web app
Open the `https://....gradio.live` link printed below (valid about 72 hours while this notebook runs).
In the app: drop an Outscraper Excel/CSV file (or pick a dataset), choose the business type and click **Analyze reviews**. Use the filters above the dashboard to explore by period, stars or theme, and the **Reply assistant** to draft answers on demand.

In [ ]:
from app import launch
launch(share=True)